# SplitSense-Bench: Can Data-Science Agents Reliably Execute Validation-Based Risk Decisions?

This notebook reconstructs the frozen Diagnostic-D2 public statistics and runs a deterministic declarative-executor demo. It uses no model API, GPU, network access, hidden labels, or private raw episodes.


In [ ]:
from pathlib import Path
import csv, json, math, random, hashlib, os

def find_data():
    candidates = [Path(os.environ['SPLITSENSE_DATA'])] if os.environ.get('SPLITSENSE_DATA') else []
    candidates += [Path('/kaggle/input/splitsense-public-artifact'), Path('../kaggle_artifact'), Path('kaggle_artifact')]
    candidates += list(Path('/kaggle/input').glob('**')) if Path('/kaggle/input').exists() else []
    for candidate in candidates:
        meta = candidate / 'metadata.json'
        if meta.is_file():
            try:
                if json.loads(meta.read_text()).get('schema') == 'splitsense-public-results-v1':
                    return candidate
            except Exception:
                pass
    raise FileNotFoundError('SplitSense public artifact not found')

DATA = find_data()
print('Public artifact:', DATA)


## Frozen conditions

- **N0 Direct:** Agent writes the full vector.
- **N1 Calculator:** Agent uses bounded arithmetic, then writes the vector.
- **N2 Declarative:** Agent locks evidence, bindings, and weights; a restricted executor produces the vector without fixing semantic errors.

Strict success `J_D2` requires a valid lock, correct semantic plan `P`, and every normalized component error at or below `1e-4`.


In [ ]:
MODES = ('N0_DIRECT_OUTPUT', 'N1_CALCULATOR', 'N2_DECLARATIVE_EXECUTION')

def mean(values): return sum(values) / len(values)
def quantile(values, p):
    values = sorted(values); pos = (len(values)-1)*p; lo, hi = math.floor(pos), math.ceil(pos)
    return values[lo] if lo == hi else values[lo]*(hi-pos) + values[hi]*(pos-lo)
def bootstrap(values, seed):
    rng = random.Random(seed)
    draws = [mean([values[rng.randrange(len(values))] for _ in values]) for _ in range(20000)]
    return 100*mean(values), [100*quantile(draws, .025), 100*quantile(draws, .975)]
def load(name):
    with (DATA/name).open(newline='') as f: rows=list(csv.DictReader(f))
    for row in rows:
        for key in ('valid_lock','semantic_plan_correct_P','end_to_end_success_J_D2'): row[key]=int(row[key])
    return rows
def analyze(rows, seed):
    modes={}
    for mode in MODES:
        selected=[r for r in rows if r['execution_mode']==mode]; wins=sum(r['end_to_end_success_J_D2'] for r in selected)
        modes[mode]=(wins,len(selected),100*wins/len(selected))
    index={(r['world'],r['deployment'],r['execution_mode']):r for r in rows}
    worlds=sorted({r['world'] for r in rows}); deployments=sorted({r['deployment'] for r in rows})
    comparisons={}
    for left,right,label in ((MODES[2],MODES[0],'N2-N0'),(MODES[1],MODES[0],'N1-N0'),(MODES[2],MODES[1],'N2-N1')):
        diffs=[mean([index[(w,d,left)]['end_to_end_success_J_D2']-index[(w,d,right)]['end_to_end_success_J_D2'] for d in deployments]) for w in worlds]
        comparisons[label]=bootstrap(diffs,seed)
    return modes,comparisons

meta=json.loads((DATA/'metadata.json').read_text())
main=analyze(load('public_results.csv'),meta['main']['bootstrap_seed'])
rep=analyze(load('replication_results.csv'),meta['replication']['bootstrap_seed'])
for label,result in [('Main',main),('Independent replication',rep)]:
    print('\n'+label)
    for mode,(wins,n,pct) in result[0].items(): print(f'  {mode}: {wins}/{n} ({pct:.2f}%)')
    est,ci=result[1]['N2-N0']; print(f'  N2-N0: {est:.2f} pp, 95% CI [{ci[0]:.2f}, {ci[1]:.2f}]')


In [ ]:
# Create a dependency-free SVG comparison chart.
batches=[('Main',main[0]),('Independent replication',rep[0])]
colors={'N0_DIRECT_OUTPUT':'#8A94A6','N1_CALCULATOR':'#D79A3B','N2_DECLARATIVE_EXECUTION':'#177E89'}
svg=['<svg xmlns="http://www.w3.org/2000/svg" width="900" height="460" viewBox="0 0 900 460">','<rect width="100%" height="100%" fill="white"/>','<text x="450" y="28" text-anchor="middle" font-family="Arial" font-size="18" font-weight="700">Strict end-to-end success</text>']
for tick in range(0,51,10):
    y=360-tick/50*280; svg += [f'<line x1="70" y1="{y}" x2="860" y2="{y}" stroke="#ddd"/>',f'<text x="60" y="{y+4}" text-anchor="end" font-family="Arial" font-size="12">{tick}%</text>']
for center,(batch,modes) in zip((270,650),batches):
    for i,mode in enumerate(MODES):
        wins,n,pct=modes[mode]; x=center-105+i*78; h=pct/50*280; y=360-h
        svg += [f'<rect x="{x}" y="{y}" width="58" height="{h}" fill="{colors[mode]}"/>',f'<text x="{x+29}" y="{max(y-7,45)}" text-anchor="middle" font-family="Arial" font-size="12" font-weight="700">{wins}/{n}</text>']
    svg.append(f'<text x="{center-27}" y="390" text-anchor="middle" font-family="Arial" font-size="14" font-weight="700">{batch}</text>')
svg.append('</svg>'); chart='\n'.join(svg); Path('splitsense_comparison.svg').write_text(chart)
try:
    from IPython.display import SVG, display
    display(SVG(data=chart))
except ImportError:
    print('Chart written to splitsense_comparison.svg')


## Deterministic declarative-executor demo

This public toy task is not a formal D2 episode. It demonstrates that a locked Agent-style plan can be executed without a model key, network, or hidden answer.


In [ ]:
task=json.loads((DATA/'task_examples.json').read_text())['examples'][0]
plan=json.loads((DATA/'executor_examples.json').read_text())['examples'][0]['plan']
assert plan['op']=='execute_plan' and len(plan['outputs'])==len(task['candidate_ids'])
values={}
for output in plan['outputs']:
    total=0.0
    for term in output['terms']:
        pool=task['pools'][term['pool_id']]; idx=task['candidate_ids'].index(term['source_candidate_id'])
        total += plan['weights'][term['weight_index']] * pool['risks'][idx]
    values[output['candidate_id']]=total
risks=[values[c] for c in task['candidate_ids']]
artifact={'status':'PASS','sealed':True,'source':'AGENT_PLUS_EXECUTOR','risks':risks}
artifact['artifact_sha256']=hashlib.sha256(json.dumps(artifact,sort_keys=True,separators=(',',':')).encode()).hexdigest()
print(json.dumps({'status':artifact['status'],'sealed':artifact['sealed'],'risk_count':len(risks),'artifact_sha256':artifact['artifact_sha256']},indent=2))


## Limitations

The result is limited to one synthetic mechanism, one fixed candidate library, one frozen model and interface, and the registered task scale. N2 is an Agent-plus-executor system. N2 was not shown superior to N1. The study does not establish real-world deployment-risk accuracy, Track A model-selection success, cross-model generalization, or cross-domain generalization. Private raw trajectories and hidden generation details are not part of this public artifact.
